# 03 · Comparar modelos de regresión

**Clases 6, 7, 10, 11 y 12:** árbol de regresión, Random Forest, XGBoost, one-hot del mes, partición temporal y métricas.

**Pregunta:** ¿podemos estimar el factor de capacidad mensual de un parque con su ubicación y el clima del mes?

In [ ]:
from pathlib import Path
import os, sys
if 'google.colab' in sys.modules:
    import subprocess
    root = Path('/content/tpo-solar')
    if not root.exists():
        subprocess.run(['git','clone','https://github.com/imelinc/tpo-solar.git',str(root)],check=True)
    os.chdir(root)
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import DATA_RAW, DATA_PROCESSED
import json
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
# La evidencia la escribe python -m src.academico. Con REENTRENAR = True se recalcula acá (tarda unos minutos).
REENTRENAR = False
ruta = DATA_PROCESSED / 'defensa_evidencia.json'
if REENTRENAR or not ruta.exists():
    from src.academico import construir
    E = construir()
else:
    E = json.loads(ruta.read_text())
R = E['regresion']
print('Elegido por validación:', R['nombre'])

In [ ]:
filas = [{'modelo':r['nombre'], 'particion':p, **r[p]} for r in R['comparacion'] for p in ['train','validacion','test']]
comparacion = pd.DataFrame(filas)
print(R['protocolo'])
print(comparacion.round(4).to_string(index=False))
px.bar(comparacion, x='modelo', y='rmse', color='particion', barmode='group',
       title='Clases 11 y 12 · RMSE por partición', labels={'rmse':'RMSE (factor de capacidad)'}).show()

## Por qué este modelo

Se elige el menor RMSE de validación (2024), antes de mirar la prueba. La lineal simple es la línea de base; la múltiple suma ubicación, clima y mes; el árbol parte los datos con preguntas; Random Forest promedia árboles; XGBoost suma árboles que corrigen el error del anterior. Los modelos de árboles no extrapolan fuera del rango de latitud y longitud de los parques de entrenamiento.

In [ ]:
imp = pd.DataFrame(R.get('importancias', []))
if len(imp):
    px.bar(imp.iloc[::-1], x='valor', y='variable', orientation='h', title='Clase 7 · Importancia de variables del modelo elegido').show()
A = pd.read_parquet(DATA_PROCESSED / 'defensa_argentina.parquet')
prueba = A[A['particion'].eq('prueba')]
fig = px.scatter(prueba, x='pred', y='factor_capacidad', opacity=.5, labels={'pred':'Estimado','factor_capacidad':'Observado'},
                 title='Clase 12 · Observado contra estimado en la prueba')
fig.add_shape(type='line', x0=0, y0=0, x1=.45, y1=.45, line=dict(dash='dot'))
fig.show()
prueba = prueba.assign(residuo=prueba['factor_capacidad'] - prueba['pred'])
px.histogram(prueba, x='residuo', nbins=40, title='Clase 12 · Residuos de la prueba (observado − estimado)').show()

## Un parque que el modelo no vio

Cinco grupos por parque, solo con años de entrenamiento. Describe el caso de un sitio nuevo. No se usó para elegir el modelo.

In [ ]:
cv = R['cv_parques']
print(pd.DataFrame(cv['folds']).round(4))
print('RMSE promedio de parque nuevo:', round(cv['rmse'], 4))